# 🌐 Simple Gradio Web App + API (Colab)

This is a minimal example: a tiny Gradio web app that also exposes a JSON API endpoint — no ML, just returns example data.

- Web UI: a textbox; click submit to get sample JSON
- API: POST to /api/predict/ with body {data: ['your text']}

Note: In Colab, use the printed proxy URL to access the app from your browser. The proxy works only during your session.


In [ ]:
# Install Gradio
!pip -q install gradio > /dev/null
import gradio as gr
print('✅ Gradio installed')


In [ ]:
# Simple demo function: echoes the text and returns example data
import time
import logging, sys
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("demo")
def demo_api(text: str = ''):
    logger.info("Request received: text=%r", text)
    resp = {
        'message': f'echo: {text}',
        'examples': [
            {'id': 1, 'caption': 'example A'},
            {'id': 2, 'caption': 'example B'},
            {'id': 3, 'caption': 'example C'}
        ],
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    logger.info("Responding with keys=%s", list(resp.keys()))
    return resp
print('✅ Demo function ready')


In [ ]:
# Build a tiny Gradio Blocks app with an explicit API route name
with gr.Blocks() as demo:
    gr.Markdown('# Simple Demo')
    inp = gr.Textbox(label='Enter any text (optional)')
    out = gr.JSON(label='Response')
    btn = gr.Button('Submit')
    # api_name defines the REST path: POST /api/predict/
    btn.click(fn=demo_api, inputs=inp, outputs=out, api_name='predict')

# Launch on port 8000 so we can use Colab proxy or the Gradio share URL
demo.launch(server_name='0.0.0.0', server_port=8000, share=True, quiet=False)


### Colab Proxy URL
Use the link below to access the app from your browser while the notebook is running.


In [ ]:
try:
    from google.colab import output as colab_output
    proxy_base = colab_output.eval_js('google.colab.kernel.proxyPort(8000)')
    if proxy_base and isinstance(proxy_base, str):
        if not proxy_base.endswith('/'):
            proxy_base += '/'
        print('🔗 Open UI:', proxy_base)
        print('🔧 API (predict):', proxy_base + 'api/predict/')
    else:
        print('Colab proxy URL unavailable.')
except Exception as e:
    print('Colab proxy not available or not in Colab:', e)


## How to call the API
This Gradio app automatically exposes a JSON API at `/api/predict/`.
- Input order matches the UI components; here there's a single `Textbox` (string).
- Send a POST with body {data: ['your text']}.

Example curl (replace `<PROXY_BASE>` with the printed URL above):
```bash
curl -s -X POST '<PROXY_BASE>api/predict/' \
  -H 'Content-Type: application/json' \
  -d '{"data":["hello world"]}' | jq
```
The JSON response contains your function's return data under `data`.
